In [2]:
import json
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

# =========================
# DEVICE CONFIGURATION
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# LOAD CLASS NAMES
# =========================
with open("models/class_names.json", "r") as f:
    class_names = json.load(f)

# =========================
# LOAD MODEL
# =========================
model = models.resnet18(weights=None)

num_features = model.fc.in_features

model.fc = nn.Linear(num_features, len(class_names))

model.load_state_dict(
    torch.load("models/best_model.pth", map_location=device)
)

model = model.to(device)

model.eval()

# =========================
# IMAGE TRANSFORM
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# =========================
# IMAGE PATH
# =========================
image_path = "images.jpg"  # put your test image name here

# =========================
# LOAD IMAGE
# =========================
image = Image.open(image_path).convert("RGB")

image = transform(image)

image = image.unsqueeze(0)

image = image.to(device)

# =========================
# PREDICTION
# =========================
with torch.no_grad():

    outputs = model(image)

    probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    confidence, predicted = torch.max(probabilities, 0)

# =========================
# RESULT
# =========================
predicted_class = class_names[predicted.item()]

print("\nPrediction Result")
print("---------------------")
print(f"Disease   : {predicted_class}")
print(f"Confidence: {confidence.item() * 100:.2f}%")

C:\Users\nabha\AppData\Local\Temp\ipykernel_7296\405809316.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("models/best_model.pth", map_location=device)



Prediction Result
---------------------
Disease   : WCLWD_DryingofLeaflets
Confidence: 70.14%
